<a href="https://colab.research.google.com/github/Thiziriinfo/Analyse-Immobilier-Paris-2023/blob/main/Analyse_Immobilier_DVF_France.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyse du marché immobilier parisien — DVF 2023

**Auteure :** Thiziri Abchiche  
**Date :** Juin 2026  
**Source des données :** [data.gouv.fr — Demandes de Valeurs Foncières](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres/)

## Objectif
Analyser les transactions immobilières à Paris en 2023 à partir des données publiques DVF.  
Explorer les prix au m², leur évolution mensuelle, leur distribution géographique et leur variation selon les arrondissements et le type de bien.

## Compétences mobilisées
`Python` `pandas` `plotly` `data cleaning` `EDA` `data visualization` `cartographie`

## 1. Import des librairies

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

print("Librairies chargées")

Librairies chargées


## 2. Chargement des données

Les données DVF (Demandes de Valeurs Foncières) sont des données publiques publiées par le gouvernement français. Elles recensent toutes les transactions immobilières en France. On utilise ici les données de Paris (département 75) pour l'année 2023.

In [35]:
import urllib.request

# Téléchargement des données DVF Paris 2023 depuis data.gouv.fr
url = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/departements/75.csv.gz"
urllib.request.urlretrieve(url, "dvf_paris_2023.csv.gz")

# Chargement du fichier compressé
df = pd.read_csv("dvf_paris_2023.csv.gz", compression="gzip", low_memory=False)
print(df.shape)
df.head()


(80993, 40)


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
0,2023-1322039,2023-01-03,1,Vente,1825000.00,89.0,NaN,RUE SAINT-DENIS,8525,75001.0,...,Local industriel. commercial ou assimilé,165.0,0.0,NaN,NaN,NaN,NaN,NaN,2.349112,48.862083
1,2023-1322040,2023-01-05,1,Vente,567000.00,51.0,NaN,RUE DE L ECHIQUIER,3084,75010.0,...,Local industriel. commercial ou assimilé,52.0,0.0,NaN,NaN,NaN,NaN,NaN,2.348283,48.871818
2,2023-1322041,2023-01-04,1,Vente,140000.00,178.0,NaN,RUE DE COURCELLES,2387,75017.0,...,Appartement,18.0,1.0,NaN,NaN,NaN,NaN,NaN,2.298612,48.884255
3,2023-1322042,2023-01-05,1,Vente,400000.00,12.0,NaN,RUE TURGOT,9508,75009.0,...,Appartement,43.0,1.0,NaN,NaN,NaN,NaN,NaN,2.345859,48.880564
4,2023-1322043,2023-01-04,1,Vente,141343.12,63.0,B,RUE DAMREMONT,2534,75018.0,...,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,2.334615,48.891525


In [32]:
# afficher les colonnes
print(df.columns.tolist())

['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation', 'valeur_fonciere', 'adresse_numero', 'adresse_suffixe', 'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune', 'nom_commune', 'code_departement', 'ancien_code_commune', 'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle', 'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero', 'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez', 'lot4_numero', 'lot4_surface_carrez', 'lot5_numero', 'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales', 'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale', 'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude']


## 3. Nettoyage et préparation des données

On sélectionne 10 colonnes utiles sur 40, on filtre sur les appartements et maisons uniquement, on supprime les valeurs manquantes et on crée la variable clé : `prix_m2 = valeur_fonciere / surface_reelle_bati`.

In [6]:
# Sélection des 10 colonnes utiles sur 40
colonnes_utiles = [
    'date_mutation',
    'nature_mutation',
    'valeur_fonciere',
    'code_postal',
    'nom_commune',
    'type_local',
    'surface_reelle_bati',
    'nombre_pieces_principales',
    'longitude',
    'latitude'
]

df_clean = df[colonnes_utiles].copy()
print(df_clean.shape)
df_clean.head()

(80993, 10)


,date_mutation,nature_mutation,valeur_fonciere,code_postal,nom_commune,type_local,surface_reelle_bati,nombre_pieces_principales,longitude,latitude
0,2023-01-03,Vente,1825000.00,75001.0,Paris 1er Arrondissement,Local industriel. commercial ou assimilé,165.0,0.0,2.349112,48.862083
1,2023-01-05,Vente,567000.00,75010.0,Paris 10e Arrondissement,Local industriel. commercial ou assimilé,52.0,0.0,2.348283,48.871818
2,2023-01-04,Vente,140000.00,75017.0,Paris 17e Arrondissement,Appartement,18.0,1.0,2.298612,48.884255
3,2023-01-05,Vente,400000.00,75009.0,Paris 9e Arrondissement,Appartement,43.0,1.0,2.345859,48.880564
4,2023-01-04,Vente,141343.12,75018.0,Paris 18e Arrondissement,Dépendance,NaN,0.0,2.334615,48.891525


In [36]:
# --- Vérification des valeurs manquantes ---
missing_nan = df_clean.isnull().sum()
missing_empty = (df_clean == "").sum()
missing_spaces = (df_clean.apply(lambda x: x.astype(str).str.strip() == "")).sum()

summary = pd.DataFrame({'NaN': missing_nan, 'Vide ""': missing_empty, 'Espaces': missing_spaces})
print(summary[summary.sum(axis=1) > 0])

# --- Exploration de type_local avant filtrage ---
print(df_clean['type_local'].value_counts())

# --- Filtre : on garde uniquement Appartement et Maison ---
df_clean = df_clean[df_clean['type_local'].isin(['Appartement', 'Maison'])].copy()

# --- Suppression des lignes avec valeurs manquantes critiques ---
df_clean = df_clean.dropna(subset=['valeur_fonciere', 'longitude', 'latitude', 'code_postal'])

# --- Création de la variable prix au m² ---
df_clean['prix_m2'] = df_clean['valeur_fonciere'] / df_clean['surface_reelle_bati']

# --- Suppression des outliers : plage réaliste pour Paris ---
df_clean = df_clean[(df_clean['prix_m2'] >= 1000) & (df_clean['prix_m2'] <= 50000)]

# --- Conversion de la date et extraction du mois ---
df_clean['date_mutation'] = pd.to_datetime(df_clean['date_mutation'])
df_clean['mois'] = df_clean['date_mutation'].dt.to_period('M')

print(f"Dataset final : {df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes")

Empty DataFrame
Columns: [NaN, Vide "", Espaces]
Index: []
type_local
Appartement    30574
Maison           167
Name: count, dtype: int64
Dataset final : 30741 lignes, 13 colonnes


In [37]:
df_clean = df_clean[df_clean['type_local'].isin(['Appartement', 'Maison'])].copy()
print(df_clean.shape)

(30741, 13)


**Dataset final :** 30 741 transactions · 0 valeur manquante · prix_m2 entre 1 000€ et 50 000€/m²

## 4. Évolution du prix médian au m² sur l'année 2023

In [42]:
import plotly.express as px

# Prix médian au m² agrégé par mois
evolution = df_clean.groupby('mois')['prix_m2'].median().reset_index()
evolution['mois'] = evolution['mois'].astype(str)

# Graphe en courbe interactive — évolution mensuelle
fig = px.line(
    evolution,
    x='mois',
    y='prix_m2',
    title="Évolution du prix médian au m² — Paris 2023",
    markers=True,
    labels={'mois': 'Mois', 'prix_m2': 'Prix médian (€/m²)'},
    color_discrete_sequence=['#E84A2F']
)

# Style des points et de la ligne
fig.update_traces(
    line=dict(width=3),
    marker=dict(size=9, color='white', line=dict(width=3, color='#E84A2F'))
)

# Mise en page
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    title=dict(font=dict(size=18, color='#7B1A1A'), x=0.5),
    xaxis=dict(showgrid=False),
    yaxis=dict(gridcolor='#EEEEEE', tickformat=',.0f', ticksuffix='€'),
    hovermode='x unified'
)

fig.show()

In [39]:
print(evolution.iloc[0])   # janvier
print(evolution.iloc[-1])  # décembre

mois            2023-01
prix_m2    10746.268657
Name: 0, dtype: object
mois       2023-12
prix_m2    10000.0
Name: 11, dtype: object


Le prix médian au m² recule de 10 746€ en janvier à 10 000€ en décembre 2023,
soit une baisse de 6.9% sur l'année.

La baisse est régulière et sans rupture brutale — pas de choc ponctuel,
c'est une correction progressive du marché.

Médiane utilisée : moins sensible aux transactions atypiques
(biens de luxe > 30 000€/m²) que la moyenne.

## 5. Prix médian au m² par arrondissement

In [43]:
# Extraction du numéro d'arrondissement depuis le code postal (ex: 75006 → 6)
df_clean['arrondissement'] = df_clean['code_postal'].astype(float).astype(int).astype(str).str[-2:].astype(int)

# Prix médian par arrondissement, trié par ordre croissant
arrond = df_clean.groupby('arrondissement')['prix_m2'].median().reset_index()
arrond = arrond.sort_values('prix_m2', ascending=True)

# Graphe en barres horizontales — couleur proportionnelle au prix
fig = px.bar(
    arrond,
    x='prix_m2',
    y='arrondissement',
    orientation='h',
    title="Prix médian au m² par arrondissement — Paris 2023",
    labels={'prix_m2': 'Prix médian (€/m²)', 'arrondissement': 'Arrondissement'},
    color='prix_m2',
    color_continuous_scale='YlOrRd',
    text='prix_m2'
)

# Affichage des valeurs sur les barres
fig.update_traces(
    texttemplate='%{text:,.0f}€',
    textposition='outside'
)

# Mise en page
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    title=dict(font=dict(size=18, color='#7B1A1A'), x=0.5),
    coloraxis_showscale=False,
    yaxis=dict(tickprefix='Paris ', dtick=1),
    xaxis=dict(showgrid=False),
    height=600
)

fig.show()


Les arrondissements les plus chers sont le 6e (15 250€/m²) et le 7e (14 971€/m²),
concentrés sur la Rive Gauche et les quartiers historiquement haut de gamme.

Les moins chers sont le 19e (8 838€/m²) et le 20e (8 860€/m²),
soit un écart de 72% entre le 6e et le 19e.

L'axe ouest (6e, 7e, 8e, 16e) est systématiquement plus cher que l'axe nord-est
(18e, 19e, 20e) — tendance structurelle du marché parisien.


## 6. Distribution des prix au m²

In [44]:
# Histogramme de la distribution des prix au m²
fig = px.histogram(
    df_clean,
    x='prix_m2',
    nbins=60,  # 60 intervalles pour une lecture fine
    title="Distribution des prix au m² — Paris 2023",
    labels={'prix_m2': 'Prix au m² (€)', 'count': 'Nombre de transactions'},
    color_discrete_sequence=['#E84A2F']
)

# Mise en page
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    title=dict(font=dict(size=18, color='#7B1A1A'), x=0.5),
    xaxis=dict(showgrid=False, ticksuffix='€'),
    yaxis=dict(gridcolor='#EEEEEE', title='Nombre de transactions'),
    bargap=0.05
)

# Ligne verticale sur la médiane pour référence
fig.add_vline(
    x=df_clean['prix_m2'].median(),
    line_dash='dash',
    line_color='#1F3864',
    line_width=2,
    annotation_text=f"Médiane : {int(df_clean['prix_m2'].median()):,}€",
    annotation_position='top right',
    annotation_font_color='#1F3864'
)

fig.show()

La distribution est asymétrique à droite : la majorité des transactions
se concentre entre 9 000€ et 11 000€/m², avec un pic autour de 10 000€/m².

La queue droite s'étend jusqu'à 50 000€/m² — ce sont les transactions
de luxe, rares mais qui tirent la moyenne vers le haut.

Médiane : 10 377€/m² — Moyenne : 11 526€/m²
L'écart de 1 149€ entre les deux confirme l'effet des valeurs extrêmes.
Sur ce dataset, la médiane est l'indicateur de référence.

## 7. Prix médian au m² selon le nombre de pièces

In [45]:
# Filtre : on exclut les 0 pièce (erreurs de saisie) et les > 6 pièces (cas rares)
pieces = df_clean[df_clean['nombre_pieces_principales'] >= 1].groupby('nombre_pieces_principales')['prix_m2'].median().reset_index()
pieces = pieces[pieces['nombre_pieces_principales'] <= 6]

# Graphe en barres — couleur proportionnelle au prix
fig = px.bar(
    pieces,
    x='nombre_pieces_principales',
    y='prix_m2',
    title="Prix médian au m² selon le nombre de pièces — Paris 2023",
    labels={'nombre_pieces_principales': 'Nombre de pièces', 'prix_m2': 'Prix médian (€/m²)'},
    color='prix_m2',
    color_continuous_scale='YlOrRd',
    text='prix_m2'
)

# Affichage des valeurs sur les barres
fig.update_traces(
    texttemplate='%{text:,.0f}€',
    textposition='outside'
)

# Mise en page
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    title=dict(font=dict(size=18, color='#7B1A1A'), x=0.5),
    coloraxis_showscale=False,
    xaxis=dict(showgrid=False, dtick=1, title='Nombre de pièces'),
    yaxis=dict(gridcolor='#EEEEEE', ticksuffix='€', range=[0, 16000]),
    height=500
)

fig.show()

### Interprétation

Le prix au m² augmente avec le nombre de pièces : de 10 175€/m² pour les studios
à 13 103€/m² pour les 6 pièces et plus.

Exception notable : les studios (1 pièce) sont plus chers que les 2  pièces.
Cela s'explique par leur localisation — les petites surfaces sont souvent situées
dans les arrondissements centraux et haut de gamme (6e, 7e, 8e).

Ce graphe illustre qu'on ne peut pas interpréter le prix au m² sans tenir compte
de la localisation — les deux variables sont interdépendantes.

## 8. Dispersion des prix par arrondissement

In [46]:
# Boxplot — dispersion des prix au m² pour chaque arrondissement
fig = px.box(
    df_clean,
    x='arrondissement',
    y='prix_m2',
    title="Dispersion des prix au m² par arrondissement — Paris 2023",
    labels={'arrondissement': 'Arrondissement', 'prix_m2': 'Prix au m² (€)'},
    color='arrondissement',
    color_discrete_sequence=px.colors.sequential.YlOrRd
)

# Mise en page
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    title=dict(font=dict(size=18, color='#7B1A1A'), x=0.5),
    xaxis=dict(showgrid=False, title='Arrondissement', dtick=1),
    yaxis=dict(gridcolor='#EEEEEE', ticksuffix='€'),
    showlegend=False,
    height=550
)

fig.show()

Le boxplot révèle la dispersion des prix au sein de chaque arrondissement.

Les arrondissements centraux (1er, 2e, 3e) affichent les boîtes les plus larges —
grande hétérogénéité des prix, coexistence de biens standards et de biens de luxe.

Les arrondissements périphériques (19e, 20e) ont des boîtes plus resserrées —
marché plus homogène, moins de transactions exceptionnelles.

Les points au-dessus des moustaches sont des outliers — transactions atypiques
(biens d'exception, erreurs de surface) présents dans tous les arrondissements.

## 9. Carte géographique des transactions

In [47]:
# Carte interactive — chaque point = une transaction, couleur = prix au m²
fig = px.scatter_mapbox(
    df_clean,
    lat='latitude',
    lon='longitude',
    color='prix_m2',
    size='prix_m2',
    color_continuous_scale='YlOrRd',
    size_max=8,
    zoom=11,
    center={"lat": 48.8566, "lon": 2.3522},  # Centre de Paris
    title="Prix au m² par transaction — Paris 2023",
    labels={'prix_m2': 'Prix au m²'},
    hover_data={'code_postal': True, 'prix_m2': ':,.0f', 'latitude': False, 'longitude': False}
)

# Fond de carte neutre pour faire ressortir les couleurs
fig.update_layout(
    mapbox_style="carto-positron",
    title=dict(font=dict(size=18, color='#7B1A1A'), x=0.5),
    font=dict(family='Arial', size=12),
    height=650,
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig.show()

La carte confirme visuellement ce que les graphes précédents montraient en chiffres.

Les points rouge foncé (prix > 20 000€/m²) se concentrent dans le centre-ouest :
6e, 7e, 8e arrondissements. Les points jaunes (prix < 10 000€/m²) dominent
la périphérie nord-est : 18e, 19e, 20e.

La densité de transactions est plus forte dans les arrondissements populaires —
plus de volume, mais à des prix plus accessibles.

## 10. Conclusion

Ce projet analyse 30 741 transactions immobilières à Paris en 2023 à partir des données publiques DVF.

**3 insights clés :**

1. **Baisse de 6.9% sur l'année** — correction progressive du marché parisien en 2023
2. **Écart de 72% entre le 6e et le 19e** — la localisation reste le premier facteur de prix
3. **La médiane (10 377€/m²) est plus représentative que la moyenne (11 526€/m²)** — le marché du luxe biaise la lecture moyenne

**Limites :**
- Données limitées à Paris intra-muros 2023
- Pas de données sur les taux d'intérêt ou le contexte macro pour expliquer les causes

